# Canonical dataset cleaning

This notebook turns the normalized source tables produced by `src/ingest.py` into five auditable analysis datasets. The workflow is deliberately repeated for every dataset: **read source data → parse/compute → validate → write CSV → read the CSV back and preview it**.

Outputs are written to `reports/analysis/data/clean/datasetA.csv` through `datasetE.csv`. The raw JSON/YAML artifacts remain owned by `src/ingest.py`; this notebook starts from its lossless, row-level CSV outputs so there is only one raw-artifact parser in the project.

## Canonical finding terminology

| Meaning | Symbol | Paper term | Canonical CSV column | Source column |
|---|---|---|---|---|
| Findings present at the current checkpoint | $A_{a,p,k}$ | Absolute Finding Count | `absolute_finding_count` | `total_findings_absolute` |
| Findings introduced since the previous checkpoint | $I^{local}_{a,p,k}$ | Task-Local Introduced Findings | `task_local_introduced_findings` | `run_local_introduced_count` |
| Findings resolved since the previous checkpoint | $R^{local}_{a,p,k}$ | Task-Local Resolved Findings | `task_local_resolved_findings` | `run_local_resolved_count` |
| Introduced minus resolved since the previous checkpoint | $N^{local}_{a,p,k}$ | Task-Local Net Change | `task_local_net_change` | `run_local_net_change` |
| Current findings absent from baseline | $I^{base}_{a,p,k}$ | Baseline-Relative Introduced Findings | `baseline_relative_introduced_findings` | `trajectory_introduced_count` |
| Baseline findings absent at the current checkpoint | $R^{base}_{a,p,k}$ | Baseline-Relative Resolved Findings | `baseline_relative_resolved_findings` | `trajectory_resolved_count` |
| Introduced minus resolved relative to baseline | $N^{base}_{a,p,k}$ | Baseline-Relative Net Change | `baseline_relative_net_change` | `trajectory_net_change` |

The word `active` is not used as a numeric column because it can ambiguously mean either the absolute count $A$ or baseline-relative net change $N^{base}$.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

def locate_analysis_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "runs.csv").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate reports/analysis from the current working directory.")

ANALYSIS_DIR = locate_analysis_dir(Path.cwd().resolve())
DATA_DIR = ANALYSIS_DIR / "data"
DERIVED_DIR = DATA_DIR / "derived"
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR, CLEAN_DIR

(PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis'),
 PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/clean'))

## 1. Read normalized source data

These are the row-level source tables generated from the original experiment artifacts by `src/ingest.py`. Derived presentation tables are not used as inputs, which keeps the five canonical datasets independently reproducible.

In [2]:
runs = pd.read_csv(DATA_DIR / "runs.csv")
constraint_findings = pd.read_csv(DATA_DIR / "constraint_findings.csv")
metric_observations = pd.read_csv(DATA_DIR / "metric_observations.csv")
task_completion = pd.read_csv(DATA_DIR / "task_completion.csv")
review_runs = pd.read_csv(DATA_DIR / "review_runs.csv")
review_findings = pd.read_csv(DATA_DIR / "review_findings.csv")

sources = {
    "runs": runs,
    "constraint_findings": constraint_findings,
    "metric_observations": metric_observations,
    "task_completion": task_completion,
    "review_runs": review_runs,
    "review_findings": review_findings,
}
display(pd.DataFrame([
    {"source_table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in sources.items()
]))
display(runs.head(3))

,source_table,rows,columns
0,runs,13,31
1,constraint_findings,689,13
2,metric_observations,247,16
3,task_completion,16,32
4,review_runs,4,8
5,review_findings,25,11


,evaluation_id,session_id,task_id,task_order,agent,strategy,model,execution_status,constraint_result,comparison_status,duration_ms,backend_status,frontend_status,cross_status,rules_evaluated,backend_findings_absolute,frontend_findings_absolute,cross_findings_absolute,metrics_total,metrics_scored,metric_errors,scope_errors,run_local_introduced_count,run_local_resolved_count,run_local_net_change,trajectory_introduced_count,trajectory_resolved_count,trajectory_net_change,source_file,total_findings_absolute,metric_coverage
0,baseline/Base,baseline,Base,0,baseline,baseline,NaN,completed,passed,valid,4475,completed,completed,completed,36,0,5,0,19,18,0,0,0,0,0,0,0,0,/Users/luowei/project/ai-architecture-integrit...,5,0.947368
1,session_20260817_130253/T1,session_20260817_130253,T1,1,claude,minimal,claude-sonnet-4-6,completed,failed,valid,4309,completed,completed,completed,36,8,8,0,19,19,0,0,11,0,11,11,0,11,/Users/luowei/project/ai-architecture-integrit...,16,1.000000
2,session_20260817_130253/T2,session_20260817_130253,T2,2,claude,minimal,claude-sonnet-4-6,completed,failed,valid,8452,completed,completed,completed,36,5,11,0,19,19,0,0,9,9,0,12,1,11,/Users/luowei/project/ai-architecture-integrit...,16,1.000000


In [3]:
CONCERN_ORDER = [
    ("BE-STRUCT", "backend"), ("BE-DEP", "backend"),
    ("BE-DOM", "backend"), ("BE-ERR", "backend"),
    ("BE-CONTRACT", "backend"), ("BE-ROUTE", "backend"),
    ("BE-SIZE", "backend"), ("BE-DUP", "backend"),
    ("BE-TEST", "backend"), ("FE-COM", "frontend"),
    ("FE-STATE", "frontend"), ("FE-ROUTE", "frontend"),
    ("FE-STYLE", "frontend"), ("FE-DATA", "frontend"),
    ("FE-COMM", "frontend"), ("FE-DUP", "frontend"),
    ("CROSS-EP", "cross-stack"), ("CROSS-TYPE", "cross-stack"),
    ("CROSS-PROP", "cross-stack"),
]
CONCERN_RANK = {concern: rank for rank, (concern, _) in enumerate(CONCERN_ORDER)}

def condition_label(frame: pd.DataFrame) -> pd.Series:
    return frame["agent"].str.title() + " " + frame["strategy"].str.title()

def concern_from_identifier(identifier: pd.Series) -> pd.Series:
    return identifier.str.extract(r"^((?:BE|FE|CROSS)-[A-Z]+)-", expand=False)

def save_and_report(frame: pd.DataFrame, filename: str) -> Path:
    path = CLEAN_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows × {len(frame.columns)} columns -> {path.relative_to(ANALYSIS_DIR)}")
    return path

def read_back(filename: str, n: int = 5) -> pd.DataFrame:
    path = CLEAN_DIR / filename
    frame = pd.read_csv(path)
    print(f"Read back {len(frame):,} rows from {path.relative_to(ANALYSIS_DIR)}")
    return frame.head(n)

## 2. Dataset A — Checkpoint Summary

Grain: one row per agent checkpoint (`session × T1–T3`). Functional acceptance is kept separate from Harness execution and architecture findings. The two identities below must hold for every row:

$$A_k = A_{k-1} + I^{local}_k - R^{local}_k$$

$$A_k = A_{base} + I^{base}_k - R^{base}_k$$

In [4]:
agent_runs = runs.loc[runs["session_id"] != "baseline"].copy()
acceptance = task_completion.loc[
    task_completion["task_id"].isin(["T1", "T2", "T3"]),
    ["session_id", "task_id", "test_status", "test_cases_total", "test_cases_passed", "test_cases_failed"],
].rename(columns={"test_status": "functional_acceptance"})

dataset_a = agent_runs.merge(acceptance, on=["session_id", "task_id"], how="left", validate="one_to_one")
dataset_a["condition"] = condition_label(dataset_a)
dataset_a = dataset_a.rename(columns={
    "task_id": "task",
    "test_cases_total": "functional_test_cases_total",
    "test_cases_passed": "functional_test_cases_passed",
    "test_cases_failed": "functional_test_cases_failed",
    "total_findings_absolute": "absolute_finding_count",
    "run_local_introduced_count": "task_local_introduced_findings",
    "run_local_resolved_count": "task_local_resolved_findings",
    "run_local_net_change": "task_local_net_change",
    "trajectory_introduced_count": "baseline_relative_introduced_findings",
    "trajectory_resolved_count": "baseline_relative_resolved_findings",
    "trajectory_net_change": "baseline_relative_net_change",
})
a_columns = [
    "evaluation_id", "condition", "session_id", "agent", "strategy",
    "task", "task_order", "execution_status", "functional_acceptance",
    "functional_test_cases_total", "functional_test_cases_passed", "functional_test_cases_failed",
    "absolute_finding_count",
    "task_local_introduced_findings", "task_local_resolved_findings", "task_local_net_change",
    "baseline_relative_introduced_findings", "baseline_relative_resolved_findings",
    "baseline_relative_net_change",
]
dataset_a = dataset_a[a_columns].sort_values(["agent", "strategy", "task_order"]).reset_index(drop=True)
finding_columns = [column for column in a_columns if column.endswith("_findings") or column.endswith("_count") or column.endswith("_change")]
dataset_a[finding_columns] = dataset_a[finding_columns].astype("int64")
functional_test_columns = ["functional_test_cases_total", "functional_test_cases_passed", "functional_test_cases_failed"]
assert dataset_a[functional_test_columns].notna().all().all()
dataset_a[functional_test_columns] = dataset_a[functional_test_columns].astype("int64")
assert dataset_a["functional_test_cases_total"].eq(
    dataset_a["functional_test_cases_passed"] + dataset_a["functional_test_cases_failed"]
).all()

baseline_absolute = int(runs.loc[runs["session_id"] == "baseline", "total_findings_absolute"].iloc[0])
previous_absolute = dataset_a.groupby("session_id")["absolute_finding_count"].shift().fillna(baseline_absolute).astype(int)
audit_a = dataset_a[["evaluation_id"]].copy()
audit_a["task_local_net_identity"] = dataset_a["task_local_net_change"].eq(
    dataset_a["task_local_introduced_findings"] - dataset_a["task_local_resolved_findings"]
)
audit_a["baseline_relative_net_identity"] = dataset_a["baseline_relative_net_change"].eq(
    dataset_a["baseline_relative_introduced_findings"] - dataset_a["baseline_relative_resolved_findings"]
)
audit_a["absolute_from_local_identity"] = dataset_a["absolute_finding_count"].eq(
    previous_absolute + dataset_a["task_local_net_change"]
)
audit_a["absolute_from_baseline_identity"] = dataset_a["absolute_finding_count"].eq(
    baseline_absolute + dataset_a["baseline_relative_net_change"]
)
assert audit_a.drop(columns="evaluation_id").all().all(), audit_a.loc[~audit_a.drop(columns="evaluation_id").all(axis=1)]
display(audit_a)
save_and_report(dataset_a, "datasetA.csv")

,evaluation_id,task_local_net_identity,baseline_relative_net_identity,absolute_from_local_identity,absolute_from_baseline_identity
0,session_20260817_130253/T1,True,True,True,True
1,session_20260817_130253/T2,True,True,True,True
2,session_20260817_130253/T3,True,True,True,True
3,session_20260817_160757/T1,True,True,True,True
4,session_20260817_160757/T2,True,True,True,True
5,session_20260817_160757/T3,True,True,True,True
6,session_20260817_210911/T1,True,True,True,True
7,session_20260817_210911/T2,True,True,True,True
8,session_20260817_210911/T3,True,True,True,True
9,session_20260817_221755/T1,True,True,True,True


Saved 12 rows × 19 columns -> data/clean/datasetA.csv


PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/clean/datasetA.csv')

In [5]:
dataset_a_check = pd.read_csv(CLEAN_DIR / "datasetA.csv")
assert dataset_a_check.equals(dataset_a)
display(read_back("datasetA.csv"))

Read back 12 rows from data/clean/datasetA.csv


,evaluation_id,condition,session_id,agent,strategy,task,task_order,execution_status,functional_acceptance,functional_test_cases_total,functional_test_cases_passed,functional_test_cases_failed,absolute_finding_count,task_local_introduced_findings,task_local_resolved_findings,task_local_net_change,baseline_relative_introduced_findings,baseline_relative_resolved_findings,baseline_relative_net_change
0,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,completed,pass,16,16,0,16,11,0,11,11,0,11
1,session_20260817_130253/T2,Claude Minimal,session_20260817_130253,claude,minimal,T2,2,completed,pass,33,33,0,16,9,9,0,12,1,11
2,session_20260817_130253/T3,Claude Minimal,session_20260817_130253,claude,minimal,T3,3,completed,fail,69,68,1,13,2,5,-3,9,1,8
3,session_20260817_160757/T1,Claude Structured,session_20260817_160757,claude,structured,T1,1,completed,pass,16,16,0,14,9,0,9,9,0,9
4,session_20260817_160757/T2,Claude Structured,session_20260817_160757,claude,structured,T2,2,completed,fail,33,7,26,14,8,8,0,10,1,9


## 3. Dataset B — Concern × Checkpoint

Grain: one row per `checkpoint × concern`. All 19 canonical concerns are present at every checkpoint, including explicit zero rows. The same seven finding terms used by Dataset A are used here. Summing B across concerns must exactly reproduce every finding measure in A.

In [6]:
finding_rows = constraint_findings.loc[constraint_findings["session_id"] != "baseline"].copy()
finding_rows["concern"] = concern_from_identifier(finding_rows["rule_id"])
assert finding_rows["concern"].notna().all()

grid = agent_runs[["evaluation_id", "session_id", "agent", "strategy", "task_id", "task_order"]].copy()
grid["condition"] = condition_label(grid)
grid = grid.merge(pd.DataFrame(CONCERN_ORDER, columns=["concern", "layer"]), how="cross")

measure_filters = {
    "absolute_finding_count": ("absolute", "current"),
    "task_local_introduced_findings": ("run_local", "introduced"),
    "task_local_resolved_findings": ("run_local", "resolved"),
    "baseline_relative_introduced_findings": ("trajectory_cumulative", "introduced"),
    "baseline_relative_resolved_findings": ("trajectory_cumulative", "resolved"),
}
for output_column, (delta_scope, change_type) in measure_filters.items():
    counts = (
        finding_rows.loc[
            finding_rows["delta_scope"].eq(delta_scope) & finding_rows["change_type"].eq(change_type)
        ]
        .groupby(["evaluation_id", "concern"]).size()
        .rename(output_column).reset_index()
    )
    grid = grid.merge(counts, on=["evaluation_id", "concern"], how="left", validate="one_to_one")

count_columns = list(measure_filters)
grid[count_columns] = grid[count_columns].fillna(0).astype("int64")
grid["task_local_net_change"] = grid["task_local_introduced_findings"] - grid["task_local_resolved_findings"]
grid["baseline_relative_net_change"] = grid["baseline_relative_introduced_findings"] - grid["baseline_relative_resolved_findings"]
grid["concern_order"] = grid["concern"].map(CONCERN_RANK)

b_columns = [
    "evaluation_id", "condition", "session_id", "agent", "strategy",
    "task_id", "task_order", "concern", "layer",
    "absolute_finding_count",
    "task_local_introduced_findings", "task_local_resolved_findings", "task_local_net_change",
    "baseline_relative_introduced_findings", "baseline_relative_resolved_findings",
    "baseline_relative_net_change",
]
dataset_b = (
    grid.sort_values(["agent", "strategy", "task_order", "concern_order"])
    .rename(columns={"task_id": "task"})
)
dataset_b = dataset_b[[column.replace("task_id", "task") for column in b_columns]].reset_index(drop=True)
assert not dataset_b.duplicated(["evaluation_id", "concern"]).any()
assert len(dataset_b) == len(dataset_a) * len(CONCERN_ORDER)

reconcile_columns = [
    "absolute_finding_count", "task_local_introduced_findings",
    "task_local_resolved_findings", "task_local_net_change",
    "baseline_relative_introduced_findings", "baseline_relative_resolved_findings",
    "baseline_relative_net_change",
]
b_totals = dataset_b.groupby("evaluation_id")[reconcile_columns].sum().sort_index()
a_totals = dataset_a.set_index("evaluation_id")[reconcile_columns].sort_index()
assert b_totals.equals(a_totals)
display(pd.DataFrame({"dataset_b_total": b_totals.sum(), "dataset_a_total": a_totals.sum()}))
save_and_report(dataset_b, "datasetB.csv")

,dataset_b_total,dataset_a_total
absolute_finding_count,250,250
task_local_introduced_findings,148,148
task_local_resolved_findings,80,80
task_local_net_change,68,68
baseline_relative_introduced_findings,198,198
baseline_relative_resolved_findings,8,8
baseline_relative_net_change,190,190


Saved 228 rows × 16 columns -> data/clean/datasetB.csv


PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/clean/datasetB.csv')

In [7]:
dataset_b_check = pd.read_csv(CLEAN_DIR / "datasetB.csv")
assert dataset_b_check.equals(dataset_b)
display(read_back("datasetB.csv", 8))

Read back 228 rows from data/clean/datasetB.csv


,evaluation_id,condition,session_id,agent,strategy,task,task_order,concern,layer,absolute_finding_count,task_local_introduced_findings,task_local_resolved_findings,task_local_net_change,baseline_relative_introduced_findings,baseline_relative_resolved_findings,baseline_relative_net_change
0,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-STRUCT,backend,0,0,0,0,0,0,0
1,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-DEP,backend,0,0,0,0,0,0,0
2,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-DOM,backend,1,1,0,1,1,0,1
3,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-ERR,backend,0,0,0,0,0,0,0
4,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-CONTRACT,backend,7,7,0,7,7,0,7
5,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-ROUTE,backend,0,0,0,0,0,0,0
6,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-SIZE,backend,0,0,0,0,0,0,0
7,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,BE-DUP,backend,0,0,0,0,0,0,0


## 4. Dataset C — Metric Observations

Grain: one row per `checkpoint × metric`. Baseline rows are used to supply `baseline_value` and are excluded from the output. Tukey statistics are computed from all available observations (baseline plus agent checkpoints), matching the current analysis methodology. `adverse_fence` is the direction-aware fence: lower for `higher_is_better`, upper for `lower_is_better`. Silent decay is true only when the metric is beyond that adverse fence and the matching concern introduced no task-local constraint finding.

In [8]:
metrics = metric_observations.copy()
metrics["concern"] = concern_from_identifier(metrics["metric_name"])
assert metrics["concern"].notna().all()

baseline_metrics = (
    metrics.loc[metrics["session_id"].eq("baseline"), ["metric_name", "value"]]
    .rename(columns={"value": "baseline_value"})
)
assert not baseline_metrics["metric_name"].duplicated().any()

bounds = (
    metrics.groupby("metric_name")["value"]
    .agg(
        n_observations="count",
        q1=lambda values: values.quantile(0.25),
        q3=lambda values: values.quantile(0.75),
    )
    .reset_index()
)
bounds["iqr"] = bounds["q3"] - bounds["q1"]
bounds["lower_fence"] = bounds["q1"] - 1.5 * bounds["iqr"]
bounds["upper_fence"] = bounds["q3"] + 1.5 * bounds["iqr"]

dataset_c = metrics.loc[metrics["session_id"] != "baseline"].merge(
    baseline_metrics, on="metric_name", how="left", validate="many_to_one"
).merge(bounds, on="metric_name", how="left", validate="many_to_one")
dataset_c["condition"] = condition_label(dataset_c)
dataset_c["baseline_relative_delta"] = dataset_c["value"] - dataset_c["baseline_value"]
delta_matches_source = np.isclose(
    dataset_c["baseline_relative_delta"],
    dataset_c["delta_trajectory_cumulative"],
    equal_nan=True,
)
assert delta_matches_source.all(), dataset_c.loc[~delta_matches_source, ["evaluation_id", "metric_name"]]
dataset_c["adverse_fence"] = np.where(
    dataset_c["direction"].eq("higher_is_better"),
    dataset_c["lower_fence"],
    dataset_c["upper_fence"],
)
dataset_c["metric_bad_outlier"] = np.where(
    dataset_c["direction"].eq("higher_is_better"),
    dataset_c["value"].lt(dataset_c["lower_fence"]),
    dataset_c["value"].gt(dataset_c["upper_fence"]),
)
constraint_failures = dataset_b[["evaluation_id", "concern", "task_local_introduced_findings"]].copy()
constraint_failures["constraint_failure"] = constraint_failures["task_local_introduced_findings"].gt(0)
dataset_c = dataset_c.merge(
    constraint_failures[["evaluation_id", "concern", "constraint_failure"]],
    on=["evaluation_id", "concern"], how="left", validate="many_to_one",
)
dataset_c["silent_decay"] = (
    dataset_c["status"].ne("error")
    & ~dataset_c["constraint_failure"]
    & dataset_c["metric_bad_outlier"]
)
dataset_c["concern_order"] = dataset_c["concern"].map(CONCERN_RANK)
dataset_c = dataset_c.rename(columns={
    "task_id": "task", "metric_name": "metric_id",
    "status": "metric_status", "value": "raw_value",
})
c_columns = [
    "evaluation_id", "condition", "session_id", "agent", "strategy",
    "task", "task_order", "scope_id", "concern", "metric_id",
    "metric_status", "raw_value", "unit", "baseline_value",
    "baseline_relative_delta", "direction", "n_observations",
    "q1", "q3", "iqr", "lower_fence", "upper_fence", "adverse_fence",
    "metric_bad_outlier", "constraint_failure", "silent_decay",
]
dataset_c = (
    dataset_c.sort_values(["agent", "strategy", "task_order", "concern_order"])
    [c_columns].reset_index(drop=True)
)
assert not dataset_c.duplicated(["evaluation_id", "metric_id"]).any()
assert len(dataset_c) == len(dataset_a) * len(CONCERN_ORDER)
display(dataset_c[["metric_bad_outlier", "constraint_failure", "silent_decay"]].sum().to_frame("count"))
save_and_report(dataset_c, "datasetC.csv")

,count
metric_bad_outlier,7
constraint_failure,42
silent_decay,3


Saved 228 rows × 26 columns -> data/clean/datasetC.csv


PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/clean/datasetC.csv')

In [9]:
dataset_c_check = pd.read_csv(CLEAN_DIR / "datasetC.csv")
assert len(dataset_c_check) == len(dataset_c)
assert list(dataset_c_check.columns) == list(dataset_c.columns)
display(read_back("datasetC.csv"))

Read back 228 rows from data/clean/datasetC.csv


,evaluation_id,condition,session_id,agent,strategy,task,task_order,scope_id,concern,metric_id,metric_status,raw_value,unit,baseline_value,baseline_relative_delta,direction,n_observations,q1,q3,iqr,lower_fence,upper_fence,adverse_fence,metric_bad_outlier,constraint_failure,silent_decay
0,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend,BE-STRUCT,BE-STRUCT-M-001-module-composition-violation-r...,pass,0.0,ratio,0.0,0.0,lower_is_better,13,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,False,False,False
1,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend,BE-DEP,BE-DEP-M-001-dependency-violation-density,pass,0.0,ratio,0.0,0.0,lower_is_better,13,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,False,False,False
2,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend,BE-DOM,BE-DOM-M-001-cross-module-deep-import-count,pass,1.0,count,0.0,1.0,lower_is_better,13,0.000000,1.0,1.000000,-1.500000,2.500000,2.500000,False,True,False
3,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend,BE-ERR,BE-ERR-M-001-exception-unification-violation-d...,pass,0.0,ratio,0.0,0.0,lower_is_better,13,0.000000,3.0,3.000000,-4.500000,7.500000,7.500000,False,False,False
4,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend,BE-CONTRACT,BE-CONTRACT-M-001-dto-validator-coverage,pass,1.0,ratio,1.0,0.0,higher_is_better,13,0.957447,1.0,0.042553,0.893618,1.063829,0.893618,False,True,False


## 5. Dataset D — File Findings

Grain: one row per file with at least one task-local introduced finding in a checkpoint. `file_share` is the file's own share; `top_file_share` is the checkpoint-level maximum $\max_f I_f / \sum_f I_f$ and is repeated across that checkpoint's rows for convenient grouping. Gini is intentionally not part of this canonical dataset.

In [10]:
introduced_by_file = constraint_findings.loc[
    constraint_findings["session_id"].ne("baseline")
    & constraint_findings["delta_scope"].eq("run_local")
    & constraint_findings["change_type"].eq("introduced")
    & constraint_findings["file"].notna()
    & constraint_findings["file"].ne(""),
].copy()
file_counts = (
    introduced_by_file.groupby(["evaluation_id", "file"]).size()
    .rename("task_local_introduced_findings").reset_index()
)
file_counts["checkpoint_introduced_findings"] = file_counts.groupby("evaluation_id")["task_local_introduced_findings"].transform("sum")
file_counts["top_file_finding_count"] = file_counts.groupby("evaluation_id")["task_local_introduced_findings"].transform("max")
file_counts["file_share"] = file_counts["task_local_introduced_findings"] / file_counts["checkpoint_introduced_findings"]
file_counts["top_file_share"] = file_counts["top_file_finding_count"] / file_counts["checkpoint_introduced_findings"]
file_counts["is_top_file"] = file_counts["task_local_introduced_findings"].eq(file_counts["top_file_finding_count"])

d_meta = dataset_a[["evaluation_id", "condition", "session_id", "agent", "strategy", "task", "task_order"]]
dataset_d = file_counts.merge(d_meta, on="evaluation_id", how="left", validate="many_to_one")
d_columns = [
    "evaluation_id", "condition", "session_id", "agent", "strategy",
    "task", "task_order", "file", "task_local_introduced_findings",
    "checkpoint_introduced_findings", "file_share", "is_top_file", "top_file_share",
]
dataset_d = dataset_d[d_columns].sort_values(
    ["agent", "strategy", "task_order", "task_local_introduced_findings", "file"],
    ascending=[True, True, True, False, True],
).reset_index(drop=True)
d_totals = dataset_d.groupby("evaluation_id")["task_local_introduced_findings"].sum().sort_index()
a_introduced = dataset_a.set_index("evaluation_id")["task_local_introduced_findings"].sort_index()
assert d_totals.equals(a_introduced.loc[d_totals.index])
assert dataset_d["top_file_share"].between(0, 1).all()
save_and_report(dataset_d, "datasetD.csv")

Saved 58 rows × 13 columns -> data/clean/datasetD.csv


PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/clean/datasetD.csv')

In [11]:
dataset_d_check = pd.read_csv(CLEAN_DIR / "datasetD.csv")
assert len(dataset_d_check) == len(dataset_d)
assert list(dataset_d_check.columns) == list(dataset_d.columns)
display(read_back("datasetD.csv"))

Read back 58 rows from data/clean/datasetD.csv


,evaluation_id,condition,session_id,agent,strategy,task,task_order,file,task_local_introduced_findings,checkpoint_introduced_findings,file_share,is_top_file,top_file_share
0,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend/src/modules/deal/deal.entity.ts,7,11,0.636364,True,0.636364
1,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,frontend/src/pages/deals/dealQueries.js,2,11,0.181818,False,0.636364
2,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,backend/src/modules/deal/deal.service.ts,1,11,0.090909,False,0.636364
3,session_20260817_130253/T1,Claude Minimal,session_20260817_130253,claude,minimal,T1,1,frontend/src/pages/deals/index.jsx,1,11,0.090909,False,0.636364
4,session_20260817_130253/T2,Claude Minimal,session_20260817_130253,claude,minimal,T2,2,backend/src/modules/deal/deal-contact.entity.ts,3,9,0.333333,True,0.333333


## 6. Dataset E — T4 Self-Assessment + Efficiency

Grain: one row per experimental condition. T4 reviews the T3 checkpoint, so the Harness value uses T3 **Absolute Finding Count**. `self_report_coverage` is self-reported findings divided by Harness absolute findings. `location_agreement` is the share of self-reported findings that mention a file also flagged by the Harness. Efficiency covers implementation tasks T1–T3 only; T4 review time/cost is excluded so the operational comparison remains aligned with the development trajectory.

In [12]:
t3 = dataset_a.loc[dataset_a["task"].eq("T3"), [
    "session_id", "condition", "agent", "strategy", "absolute_finding_count"
]]
review_summary = review_runs.rename(columns={"n_findings": "self_reported_finding_count"}).merge(
    t3, on=["session_id", "agent", "strategy"], how="left", validate="one_to_one"
)
review_summary["self_report_coverage"] = np.where(
    review_summary["absolute_finding_count"].gt(0),
    review_summary["self_reported_finding_count"] / review_summary["absolute_finding_count"],
    np.nan,
)

harness_files = constraint_findings.loc[
    constraint_findings["task_id"].eq("T3")
    & constraint_findings["delta_scope"].eq("absolute")
    & constraint_findings["file"].notna(),
    ["session_id", "file"],
].copy()
harness_files["basename"] = harness_files["file"].map(lambda value: Path(value).name)
harness_file_sets = harness_files.groupby("session_id")["basename"].agg(set).to_dict()
review_detail = review_findings.copy()
review_detail["location_match"] = review_detail.apply(
    lambda row: any(
        Path(path.strip()).name in harness_file_sets.get(row["session_id"], set())
        for path in str(row["referenced_files"] or "").split(";") if path.strip()
    ),
    axis=1,
)
location_summary = (
    review_detail.groupby("session_id")["location_match"]
    .agg(location_matches="sum", location_observations="size", location_agreement="mean")
    .reset_index()
)

development = task_completion.loc[task_completion["task_id"].isin(["T1", "T2", "T3"])].copy()
efficiency = development.groupby(["session_id", "agent", "strategy"], as_index=False).agg(
    development_task_count=("task_id", "nunique"),
    development_duration_seconds=("duration_seconds", "sum"),
    development_cost_usd=("cost_usd", "sum"),
    cost_basis=("cost_basis", lambda values: values.iloc[0] if values.nunique(dropna=False) == 1 else "mixed"),
)
efficiency["development_duration_minutes"] = efficiency["development_duration_seconds"] / 60

dataset_e = (
    review_summary.merge(location_summary, on="session_id", how="left", validate="one_to_one")
    .merge(efficiency, on=["session_id", "agent", "strategy"], how="left", validate="one_to_one")
)
dataset_e["review_checkpoint"] = "T4"
dataset_e["harness_checkpoint"] = "T3"
e_columns = [
    "condition", "session_id", "agent", "strategy",
    "review_checkpoint", "harness_checkpoint", "review_status",
    "absolute_finding_count", "self_reported_finding_count", "self_report_coverage",
    "location_matches", "location_observations", "location_agreement",
    "development_task_count", "development_duration_seconds",
    "development_duration_minutes", "development_cost_usd", "cost_basis",
]
dataset_e = dataset_e[e_columns].sort_values(["agent", "strategy"]).reset_index(drop=True)
assert dataset_e["development_task_count"].eq(3).all()
assert dataset_e["self_report_coverage"].between(0, 1).all()
assert dataset_e["location_agreement"].between(0, 1).all()
save_and_report(dataset_e, "datasetE.csv")

Saved 4 rows × 18 columns -> data/clean/datasetE.csv


PosixPath('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis/data/clean/datasetE.csv')

In [13]:
dataset_e_check = pd.read_csv(CLEAN_DIR / "datasetE.csv")
assert len(dataset_e_check) == len(dataset_e)
assert list(dataset_e_check.columns) == list(dataset_e.columns)
display(read_back("datasetE.csv"))

Read back 4 rows from data/clean/datasetE.csv


,condition,session_id,agent,strategy,review_checkpoint,harness_checkpoint,review_status,absolute_finding_count,self_reported_finding_count,self_report_coverage,location_matches,location_observations,location_agreement,development_task_count,development_duration_seconds,development_duration_minutes,development_cost_usd,cost_basis
0,Claude Minimal,session_20260817_130253,claude,minimal,T4,T3,issues_found,13,9,0.692308,2,9,0.222222,3,4058.811,67.646850,16.416497,reported
1,Claude Structured,session_20260817_160757,claude,structured,T4,T3,issues_found,13,5,0.384615,1,5,0.200000,3,4347.199,72.453317,16.435731,reported
2,Codex Minimal,session_20260817_210911,codex,minimal,T4,T3,issues_found,38,6,0.157895,5,6,0.833333,3,2030.079,33.834650,9.651673,estimated_gpt-5.3-codex_pricing
3,Codex Structured,session_20260817_221755,codex,structured,T4,T3,issues_found,24,5,0.208333,4,5,0.800000,3,2200.250,36.670833,10.966051,estimated_gpt-5.3-codex_pricing


## 7. Final output audit

The final cell re-reads every deliverable and checks filenames, row counts, column counts, duplicate keys, and the two cross-dataset reconciliations (B → A and D → A).

In [14]:
output_keys = {
    "datasetA.csv": ["evaluation_id"],
    "datasetB.csv": ["evaluation_id", "concern"],
    "datasetC.csv": ["evaluation_id", "metric_id"],
    "datasetD.csv": ["evaluation_id", "file"],
    "datasetE.csv": ["session_id"],
}
manifest_rows = []
for filename, key in output_keys.items():
    output = pd.read_csv(CLEAN_DIR / filename)
    manifest_rows.append({
        "dataset": filename,
        "rows": len(output),
        "columns": len(output.columns),
        "duplicate_keys": int(output.duplicated(key).sum()),
        "path": str((CLEAN_DIR / filename).relative_to(ANALYSIS_DIR)),
    })
output_manifest = pd.DataFrame(manifest_rows)
assert output_manifest["duplicate_keys"].eq(0).all()
display(output_manifest)
print("All five canonical datasets were generated, read back, and validated successfully.")

,dataset,rows,columns,duplicate_keys,path
0,datasetA.csv,12,19,0,data/clean/datasetA.csv
1,datasetB.csv,228,16,0,data/clean/datasetB.csv
2,datasetC.csv,228,26,0,data/clean/datasetC.csv
3,datasetD.csv,58,13,0,data/clean/datasetD.csv
4,datasetE.csv,4,18,0,data/clean/datasetE.csv


All five canonical datasets were generated, read back, and validated successfully.
